Import Global setting

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from itertools import product

RANDOM_STATE = 42

Load train/val/test features

In [ ]:
train_feat = pd.read_parquet("data/train_features.parquet")
val_feat = pd.read_parquet("data/val_features.parquet")
full_train_feat = pd.read_parquet("data/full_train_features.parquet")
test_feat = pd.read_parquet("data/full_test_features.parquet")

Feature selection

In [ ]:
# These columns are not used as model inputs.
# srch_id and prop_id are kept separately for grouping/submission.
NON_FEATURE_COLS = [
    "srch_id",
    "prop_id",
    "date_time",
    "click_bool",
    "booking_bool",
    "gross_bookings_usd",
    "position",
    "relevance"
]

# Keep only numeric columns as model features.
feature_cols = [
    col for col in train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(train_feat[col])
]

# Make sure validation has exactly the same features.
missing_in_val = set(feature_cols) - set(val_feat.columns)
extra_in_val = set(val_feat.columns) - set(train_feat.columns)

print("Number of features:", len(feature_cols))
print("Missing in validation:", missing_in_val)
print("Extra in validation:", len(extra_in_val))

print(feature_cols[:50])

Preparing Ranking Model Inputs

In [ ]:
# LightGBM ranker needs rows sorted by search group to identify which rows belong to the same search.
train_feat = train_feat.sort_values("srch_id").reset_index(drop=True)
val_feat = val_feat.sort_values("srch_id").reset_index(drop=True)

X_train = train_feat[feature_cols]
y_train = train_feat["relevance"].astype(int)

X_val = val_feat[feature_cols]
y_val = val_feat["relevance"].astype(int)

# Group sizes are needed for LightGBM ranker to know how many rows belong to each search group.
group_train = train_feat.groupby("srch_id").size().to_numpy()
group_val = val_feat.groupby("srch_id").size().to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Number of train groups:", len(group_train))
print("Number of validation groups:", len(group_val))
print("First 10 group sizes:", group_train[:10])

Evaluate the ranking quality of the model using NDCG@k metric.

In [ ]:
def dcg_at_k(relevances, k=5):
    """
    Computes DCG@k for one ranked list.
    """
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(relevances) + 2))
    gains = (2 ** relevances - 1)
    return np.sum(gains / discounts)


def ndcg_at_k_for_group(y_true, y_score, k=5):
    """
    Computes NDCG@k for one search group.
    """
    order = np.argsort(-y_score)
    ranked_relevance = y_true[order]

    ideal_order = np.argsort(-y_true)
    ideal_relevance = y_true[ideal_order]

    dcg = dcg_at_k(ranked_relevance, k=k)
    idcg = dcg_at_k(ideal_relevance, k=k)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def mean_ndcg_at_k(df, y_true_col, y_score_col, group_col="srch_id", k=5):
    """
    Computes mean NDCG@k over all searches.
    """

    scores = []

    for _, group in df.groupby(group_col):
        y_true = group[y_true_col].to_numpy()
        y_score = group[y_score_col].to_numpy()

        scores.append(ndcg_at_k_for_group(y_true, y_score, k=k))

    return np.mean(scores)

Binary Classification model - LGBMClassifier Model

In [ ]:
# ---------------------------------------------------
# Binary classification target
# ---------------------------------------------------

y_train_cls = (
    (train_feat["click_bool"] == 1) |
    (train_feat["booking_bool"] == 1)
).astype(int)

y_val_cls = (
    (val_feat["click_bool"] == 1) |
    (val_feat["booking_bool"] == 1)
).astype(int)

# ---------------------------------------------------
# Train classifier
# ---------------------------------------------------
param_grid = {
    "num_leaves": [31, 63],
    "learning_rate": [0.05, 0.03],
    "min_child_samples": [50, 100]
}

keys = list(param_grid.keys())

experiments = [
    dict(zip(keys, values))
    for values in product(*param_grid.values())
]

classifier_results = []

for i, params in enumerate(experiments, start=1):
    print(f"Training classifier {i}/{len(experiments)}")
    print(params)

    classifier = lgb.LGBMClassifier(
        objective="binary",

        n_estimators=1000,

        subsample=0.8,
        colsample_bytree=0.8,

        random_state=RANDOM_STATE,
        n_jobs=-1,

        **params
    )

classifier.fit(
    X_train,
    y_train_cls,

    eval_set=[(X_val, y_val_cls)],
    eval_metric="auc",

    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(100)
    ]
)

# ---------------------------------------------------
# Predict probabilities
# ---------------------------------------------------

preds = classifier.predict_proba(
        X_val,
        num_iteration=classifier.best_iteration_
    )[:, 1]


# ---------------------------------------------------
# Evaluate ranking quality using NDCG
# ---------------------------------------------------

val_classifier_eval = val_feat[
        ["srch_id", "prop_id", "relevance"]
    ].copy()

val_classifier_eval["prediction"] = preds

classifier_ndcg = mean_ndcg_at_k(
    val_classifier_eval,
    y_true_col="relevance",
    y_score_col="prediction",
    group_col="srch_id",
    k=5
)

classifier_results.append({
        "experiment": i,

        "num_leaves": params["num_leaves"],
        "learning_rate": params["learning_rate"],
        "min_child_samples": params["min_child_samples"],

        "best_iteration": classifier.best_iteration_,
        "validation_ndcg@5": classifier_ndcg
    })

classifier_results = pd.DataFrame(classifier_results)

classifier_results = classifier_results.sort_values(
    "validation_ndcg@5",
    ascending=False
)

best_classifier_parameters = {
    "num_leaves": int(classifier_results.iloc[0]["num_leaves"]),
    "learning_rate": float(classifier_results.iloc[0]["learning_rate"]),
    "min_child_samples": int(classifier_results.iloc[0]["min_child_samples"]),
}

print(best_classifier_parameters)

display(classifier_results)

LGBMRanker Model

In [ ]:
# ---------------------------------------------------
# Hyperparameter grid
# ---------------------------------------------------

param_grid = {
    "num_leaves": [31, 63, 127],
    "learning_rate": [0.05, 0.01, 0.1],
    "min_child_samples": [50, 100, 200]
}

# Create all parameter combinations
keys = list(param_grid.keys())

experiments = [
    dict(zip(keys, values))
    for values in product(*param_grid.values())
]

print("Total experiments:", len(experiments))

# ---------------------------------------------------
# Run experiments
# ---------------------------------------------------

tuning_results = []

for i, params in enumerate(experiments, start=1):

    print("=" * 60)
    print(f"Experiment {i}/{len(experiments)}")
    print(params)

    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[5],
        boosting_type="gbdt",

        n_estimators=1000,

        subsample=0.8,
        colsample_bytree=0.8,

        random_state=RANDOM_STATE,
        n_jobs=-1,

        **params
    )

    ranker.fit(
        X_train,
        y_train,
        group=group_train,

        eval_set=[(X_val, y_val)],
        eval_group=[group_val],
        eval_at=[5],

        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100)
        ]
    )

    # -------------------------
    # Predict validation scores
    # -------------------------

    preds = ranker.predict(
        X_val,
        num_iteration=ranker.best_iteration_
    )

    # -------------------------
    # Compute validation NDCG@5
    # -------------------------

    tmp = val_feat[["srch_id", "prop_id", "relevance"]].copy()

    tmp["prediction"] = preds

    score = mean_ndcg_at_k(
        tmp,
        y_true_col="relevance",
        y_score_col="prediction",
        group_col="srch_id",
        k=5
    )

    # -------------------------
    # Save results
    # -------------------------

    tuning_results.append({
        "experiment": i,

        "num_leaves": params["num_leaves"],
        "learning_rate": params["learning_rate"],
        "min_child_samples": params["min_child_samples"],

        "best_iteration": ranker.best_iteration_,
        "validation_ndcg@5": score
    })

# ---------------------------------------------------
# Final results table
# ---------------------------------------------------

tuning_results = pd.DataFrame(tuning_results)

tuning_results = tuning_results.sort_values(
    "validation_ndcg@5",
    ascending=False
)

best_parameters = {
    "num_leaves": int(tuning_results.iloc[0]["num_leaves"]),
    "learning_rate": float(tuning_results.iloc[0]["learning_rate"]),
    "min_child_samples": int(tuning_results.iloc[0]["min_child_samples"])
}

display(tuning_results)

In [ ]:
# Rebuild feature list from full training data.
final_feature_cols = [
    col for col in full_train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(full_train_feat[col])
]

# Ensure test has all final feature columns.
missing_in_test = [col for col in final_feature_cols if col not in test_feat.columns]
print("Missing features in test:", missing_in_test)

# Sort by srch_id to ensure correct grouping for LightGBM ranker.
full_train_feat = full_train_feat.sort_values("srch_id").reset_index(drop=True)
test_feat = test_feat.sort_values("srch_id").reset_index(drop=True)

X_full = full_train_feat[final_feature_cols]
y_full = full_train_feat["relevance"].astype(int)
group_full = full_train_feat.groupby("srch_id").size().to_numpy()

X_test = test_feat[final_feature_cols]

print("X_full:", X_full.shape)
print("X_test:", X_test.shape)
print("Number of final features:", len(final_feature_cols))

In [ ]:
# Use the best iteration from validation if available.
# If early stopping stopped at e.g. 350 trees, train final model with that many trees.
best_cls_n_estimators = ranker.best_iteration_

final_classifier = lgb.LGBMClassifier(
    objective="binary",

    n_estimators=best_cls_n_estimators,
    num_leaves=best_classifier_parameters["num_leaves"],
    learning_rate=best_classifier_parameters["learning_rate"],
    min_child_samples=best_classifier_parameters["min_child_samples"],

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=RANDOM_STATE,
    n_jobs=-1
)

In [ ]:
# Use the best iteration from validation if available.
# If early stopping stopped at e.g. 350 trees, train final model with that many trees.
best_n_estimators = ranker.best_iteration_

print("Training final model with n_estimators =", best_n_estimators)

final_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[5],
    boosting_type="gbdt",

    n_estimators=best_n_estimators,
    learning_rate=best_parameters["learning_rate"],
    num_leaves=best_parameters["num_leaves"],
    max_depth=-1,
    min_child_samples=best_parameters["min_child_samples"],
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_ranker.fit(
    X_full,
    y_full,
    group=group_full
)

In [ ]:
test_scores = final_ranker.predict(test_feat)

submission = test_feat[["srch_id", "prop_id"]].copy()
submission["score"] = test_scores

# Sort hotels within each search by predicted score descending.
submission = submission.sort_values(
    ["srch_id", "score"],
    ascending=[True, False]
)

# Required Kaggle format:
# SearchId,PropertyId
submission = submission.rename(columns={
    "srch_id": "SearchId",
    "prop_id": "PropertyId"
})

submission = submission[["SearchId", "PropertyId"]]

display(submission.head(30))

submission_path = "submission_lgbm_ranker.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
print("Submission shape:", submission.shape)

In [ ]:
final_feature_importance = pd.DataFrame({
    "feature": final_feature_cols,
    "importance": final_ranker.feature_importances_
}).sort_values("importance", ascending=False)

display(final_feature_importance.head(50))

final_feature_importance.to_csv("feature_importance_lgbm_ranker.csv", index=False)
print("Saved feature importance to feature_importance_lgbm_ranker.csv")